# ML-06 — Signal Audit: Do the Flags Hold?

**Lane 2 (refresh / opportunity scoring).** This assignment asks one question per signal:
*does the data actually show the story people tell?* I test three candidate beliefs about what
drives page decline, then test a signal one of FlyRank's own rule assumptions relies on — each
with an honest **CONFIRMED / OPPOSITE / MIXED / FALSE** verdict, visible `n`s under every
number, and a ~50-row sample-size floor.

Built with the `auditing-signals` skill on the same 108,254-page feature vector as ML-05/07.

**The one-line claim:** position is the cleanest signal (off-page-1 pages decline more); age is
**mixed** (middle-aged pages decline most, very old and very new less); and "longer pages get
more traffic" is largely a myth in this data — a log transform flips how it looks, which is
exactly why heavy-tail care matters.

## 1. Distributions

Web/traffic metrics are heavy-tailed: a few giants, a long tail of tiny values. Confirm that
before correlating anything.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath("../scripts"))
import duckdb, pandas as pd, numpy as np
from datetime import timedelta
from scipy.stats import spearmanr, pearsonr
import hf_query

con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '" + hf_query.get_token() + "')")
REL = hf_query.REL
T = {
    "content": f"read_parquet('{REL}/dim_content.parquet')",
    "daily":   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
D_MAX = con.sql(f"SELECT MAX(report_date) FROM {T['daily']}").fetchone()[0]
t = D_MAX - timedelta(days=30); b_lo = t - timedelta(days=30)

feat = con.sql(f"""
WITH win AS (
    SELECT client_hash_id, content_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position, gsc_data_available
    FROM {T['daily']} WHERE month IN ('{t:%Y-%m}', '{D_MAX:%Y-%m}')
),
agg AS (
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_b,
           SUM(CASE WHEN report_date >  DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_f,
           SUM(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available THEN 1 ELSE 0 END) AS gsc_days_b,
           AVG(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available AND gsc_avg_position IS NOT NULL AND gsc_avg_position <> 0 THEN gsc_avg_position END) AS pos_avg_b,
           STDDEV(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available AND gsc_avg_position IS NOT NULL AND gsc_avg_position <> 0 THEN gsc_avg_position END) AS pos_vol_b
    FROM win GROUP BY 1, 2
),
j AS (
    SELECT a.client_hash_id, a.content_hash_id, a.imp_b, a.imp_f, a.gsc_days_b, a.pos_avg_b, a.pos_vol_b,
           c.content_type, c.word_count, c.search_volume, c.backlinks, c.content_created_date
    FROM agg a LEFT JOIN {T['content']} c USING (client_hash_id, content_hash_id)
)
SELECT *, DATE '{t}' - content_created_date AS age_days,
       CASE WHEN imp_f < 0.8 * imp_b THEN 1 ELSE 0 END AS declined_30d
FROM j WHERE imp_b >= 100 AND gsc_days_b >= 15
""").df()

print("eligible pages:", len(feat), "| base rate:", round(feat.declined_30d.mean(), 4))
print("Heavy-tail check (min / p50 / p99 / max):")
for c in ["imp_b", "pos_avg_b", "age_days", "word_count", "search_volume", "backlinks"]:
    print(f"  {c:14s} min {np.nanmin(feat[c]):.3g}  p50 {np.nanmedian(feat[c]):.3g}  p99 {np.nanpercentile(feat[c],99):.3g}  max {np.nanmax(feat[c]):.3g}")

C:\Users\Bogdan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


eligible pages: 108254 | base rate: 0.6744
Heavy-tail check (min / p50 / p99 / max):
  imp_b          min 100  p50 593  p99 2.89e+04  max 5.96e+05
  pos_avg_b      min 0.895  p50 15  p99 71.3  max 92.2
  age_days       min 13  p50 181  p99 528  max 555
  word_count     min 0  p50 2.81e+03  p99 6.74e+03  max 1.16e+04
  search_volume  min 0  p50 10  p99 1.9e+03  max 2.46e+05
  backlinks      min 0  p50 0  p99 2.84e+03  max 2.14e+06


**Heavy tails are real:** `imp_b` spans 100 → 596k (p50 593, p99 29k) and position spans ~1 → 92.
Any plain (Pearson) correlation on raw values will be dominated by the giants — so the tests below
use **tiered medians** (what humans read best) and **rank-based (Spearman)** checks, and I show the
log-transform trap explicitly in Test 3.

## 2. Signal test #1 — "Older pages are more likely to decline"

Check: group decline rate by content age tier. Sample floor ~50 rows per tier.

In [2]:
feat["age_b"] = pd.cut(feat.age_days, [0, 90, 180, 365, 1e9], labels=["<90d", "90-180d", "181-365d", ">365d"])
t1 = feat.groupby("age_b", observed=True).agg(n=("declined_30d", "size"), decl=("declined_30d", "mean"))
print(t1)
print("\nSpearman(age_days, declined):", round(spearmanr(feat.age_days, feat.declined_30d).statistic, 3))

              n      decl
age_b                    
<90d      27280  0.587537
90-180d   26736  0.688285
181-365d  35983  0.736376
>365d     18255  0.661572

Spearman(age_days, declined): 0.084


**Verdict: MIXED.** Decline rises steadily from <90d (0.59) to 181-365d (0.74), but then
**drops** for >365d (0.66). The story "older always declines more" is only half right — there's
a vulnerable middle age (≈6-12 months), not a monotonic curve. Watching only the very oldest
pages would miss the highest-risk band.

## 2. Signal test #2 — "Pages off page 1 are more likely to decline"

Check: decline rate page-1 (avg position ≤ 10) vs off-page-1.

In [3]:
place = np.where(feat.pos_avg_b <= 10, "page_1", "off_page_1")
t2 = feat.groupby(place).agg(n=("declined_30d", "size"), decl=("declined_30d", "mean"), med_pos=("pos_avg_b", "median"))
print(t2)
print("\nSpearman(pos_avg_b, declined):", round(spearmanr(feat.pos_avg_b, feat.declined_30d).statistic, 3))

                n      decl    med_pos
off_page_1  73418  0.705740  22.109667
page_1      34836  0.608279   7.273788

Spearman(pos_avg_b, declined): 0.105


**Verdict: CONFIRMED.** Off-page-1 pages decline at 0.71 vs 0.61 for page-1 pages, and the
rank correlation is the strongest of any signal (+0.11). Being off page 1 is a genuine risk flag.
(Caveat: both groups exceed 50% because the overall base rate is 67% — position shifts *relative*
risk, it doesn't create decline from nothing.)

## 2. Signal test #3 — "Longer pages get more traffic"

A popular belief worth testing. Because traffic is heavy-tailed, compare **raw vs log** correlation
first (the trap), then read median impressions by length tier.

In [4]:
print("Pearson (raw)  imp_b ~ word_count:", round(pearsonr(feat.imp_b, feat.word_count.fillna(0))[0], 3))
print("Pearson (log1p) imp_b ~ word_count:", round(pearsonr(np.log1p(feat.imp_b), np.log1p(feat.word_count.fillna(0)))[0], 3))

feat["wc_b"] = pd.cut(feat.word_count.fillna(0), [0, 1000, 2000, 3500, 1e9], labels=["<1k", "1-2k", "2-3.5k", ">3.5k"])
t3 = feat.groupby("wc_b", observed=True).agg(n=("imp_b", "size"), med_imp=("imp_b", "median"), decl=("declined_30d", "mean"))
print(t3)

Pearson (raw)  imp_b ~ word_count: 0.046
Pearson (log1p) imp_b ~ word_count: 0.23
            n  med_imp      decl
wc_b                            
<1k       135    630.0  0.548148
1-2k     5879    303.0  0.688382
2-3.5k  64521    863.0  0.618574
>3.5k   13444    611.0  0.768968


**Verdict: MIXED / FALSE.** A raw Pearson says almost nothing (0.05); log-transformed it
jumps to 0.23 — the log is where the real (weak, non-linear) relationship lives. But the tiered
medians show **no monotonic "more is better"**: median impressions go 1-2k = 303, 2-3.5k = 863,
>3.5k = 611. There is an optimal band (≈2-3.5k words), not a longer-is-better slope. Note also
the >3.5k tier has the *highest* decline rate (0.77) — very long pages look unhealthy here.
(The <1k tier has only 135 rows; treat it as exploratory, below the comfortable floor.)

## 3. The flag-linked test

FlyRank's `position_tier` / refresh logic leans on an assumption: pages sitting **off page 1
(position > 10)** are the ones flagged for attention. I test that assumption against decline, and
re-run it on a different slice (old pages only) to check the signal survives — a real signal does,
noise doesn't.

In [5]:
def place(x):
    return np.where(x.pos_avg_b <= 10, "page_1", "off_page_1")

print("All pages:")
print(feat.groupby(place(feat)).agg(n=("declined_30d", "size"), decl=("declined_30d", "mean")).to_string())
print("\nCross-check — old pages only (age >= 180):")
old = feat[feat.age_days >= 180]
print(old.groupby(place(old)).agg(n=("declined_30d", "size"), decl=("declined_30d", "mean")).to_string())

All pages:


                n      decl
off_page_1  73418  0.705740
page_1      34836  0.608279

Cross-check — old pages only (age >= 180):
                n      decl
off_page_1  39504  0.727648
page_1      14746  0.667367


**Verdict: CONFIRMED, and it survives.** Off-page-1 declines more on the full population
(0.71 vs 0.61) and again on the old-pages cross-check (0.73 vs 0.67). A rule built on "position
slipping → refresh" has data behind it. The flag-linked assumption checks out.

## 4. What this means in practice

1. **Position is the signal to trust.** Off-page-1 status is a genuine, stable decline risk and
   matches FlyRank's own rule assumption — worth keeping first in the feature set and any refresh
   rule.
2. **Age is a trap if read as "older = riskier".** The risk peaks at ~6-12 months; refresh
   priority shouldn't be a straight "oldest first" list.
3. **Don't chase "longer pages".** There's an optimal length band (≈2-3.5k words), and very long
   pages trend *toward* decline — not the popular story. And always transform traffic-like metrics
   (log) before comparing, or the giants dominate the math.

These are observed, directional findings on this pseudonymized slice — decision-support, not
causality.

## Self-check

- [x] Distributions shown; heavy tails confirmed before correlating
- [x] traffic-like metrics handled with tiered medians / Spearman / log (Test 3 trap shown)
- [x] One mini-test per signal with a verdict (CONFIRMED / OPPOSITE / MIXED / FALSE) + n's shown
- [x] Sample-size floor respected (noted where n < ~50: the <1k word tier)
- [x] Flag-linked test re-run on a second slice to check the signal survives
- [x] No client names, URLs, or raw identifiers in any output
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] The notebook runs top to bottom with no errors
- [x] Committed to `work/notebooks/` — then submit repo URL on the ML-06 card